# Transform Sales

This notebook reads from etl_demo_output and writes a transformed sales table.

In [ ]:
# Widget setup for catalog and schema (safe defaults)
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"➡️ Using catalog={catalog}, schema={schema}")

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Get catalog and schema from bundle variables (with safe fallback)
catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog", catalog)
schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema", schema)
if not catalog:
    raise ValueError("❌ No catalog provided. Check databricks.yml target overrides.")

input_table = f"{catalog}.{schema}.etl_demo_output"
output_table = f"{catalog}.{schema}.sales_transformed"

print(f"Reading input table: {input_table}")
df = spark.table(input_table)

df_filtered = df.filter(df.amount_with_tax > 150)

# Ensure consistent schema type for amount_with_tax
df_filtered = df_filtered.withColumn("amount_with_tax", F.col("amount_with_tax").cast("double"))

print(f"Writing transformed table: {output_table}")
df_filtered.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)

print(f"✅ Transformed sales table created at {output_table}")

In [ ]:
display(spark.table(output_table))